In [4]:
pip install tensorflow==2.15.0 tensorflow-recommenders==0.7.3 tensorflow-datasets mlflow pandas numpy

  Using cached tensorflow-2.15.0-cp311-cp311-win_amd64.whl.metadata (3.6 kB)
  Using cached tensorflow_recommenders-0.7.3-py3-none-any.whl.metadata (4.6 kB)
  Using cached tensorflow_datasets-4.9.9-py3-none-any.whl.metadata (11 kB)
  Using cached mlflow-3.6.0-py3-none-any.whl.metadata (31 kB)
  Using cached pandas-2.3.3-cp311-cp311-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.3.4-cp311-cp311-win_amd64.whl.metadata (60 kB)
  Using cached tensorflow_intel-2.15.0-cp311-cp311-win_amd64.whl.metadata (5.1 kB)
  Using cached absl_py-2.3.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.9.23-py2.py3-none-any.whl.metadata (875 bytes)
  Using cached gast-0.6.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached h5py-3.15.1-cp311-cp311-win_amd64.whl.metadata (3.1 kB)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

In [5]:
import tensorflow as tf
import tensorflow_recommenders as tfrs
import tensorflow_datasets as tfds
import mlflow
import pandas as pd
import numpy as np
import os
from datetime import datetime

# Configuration MLFlow
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("news-recommender-two-tower")

2025/11/16 14:54:06 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2025/11/16 14:54:06 INFO mlflow.store.db.utils: Updating database tables
2025-11-16 14:54:06 INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
2025-11-16 14:54:06 INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
2025-11-16 14:54:06 INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
2025-11-16 14:54:06 INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


<Experiment: artifact_location='file:///c:/Users/ahmed/Rec_project/Session-Based-News-Recommendation/ml/mlruns/1', creation_time=1763294108696, experiment_id='1', last_update_time=1763294108696, lifecycle_stage='active', name='news-recommender-two-tower', tags={}>

In [25]:
# Charger
ratings = tfds.load("movielens/100k-ratings", split="train")
movies = tfds.load("movielens/100k-movies", split="train")

# Extraire  les champs 
def select_ratings_features(x):
    return {
        "user_id": x["user_id"],
        "movie_title": x["movie_title"]
    }

def select_movie_features(x):
    return {
        "movie_title": x["movie_title"]
    }

# Appliquer la sélection sur les DEUX datasets
ratings = ratings.map(select_ratings_features)
movies = movies.map(select_movie_features)

# Split train/test
tf.random.set_seed(42)
shuffled = ratings.shuffle(100_000, seed=42, reshuffle_each_iteration=False)
train_data = shuffled.take(80_000)
test_data = shuffled.skip(80_000).take(20_000)

In [26]:

# Extraire les IDs uniques 
user_ids_list = []
movie_titles_list = []

for x in ratings.batch(1000):
    user_ids_list.append(x["user_id"].numpy())
    movie_titles_list.append(x["movie_title"].numpy())

unique_user_ids = np.unique(np.concatenate(user_ids_list))
unique_movie_titles = np.unique(np.concatenate(movie_titles_list))

print(f" {len(unique_user_ids)} utilisateurs uniques")
print(f" {len(unique_movie_titles)} films uniques")

 943 utilisateurs uniques
 1664 films uniques


In [27]:
# Hyperparamètres
EMBEDDING_DIM = 64

# Tour utilisateur
user_model = tf.keras.Sequential([
    tf.keras.layers.StringLookup(
        vocabulary=unique_user_ids, mask_token=None
    ),
    tf.keras.layers.Embedding(len(unique_user_ids) + 1, EMBEDDING_DIM)
], name="user_tower")

# Tour film
movie_model = tf.keras.Sequential([
    tf.keras.layers.StringLookup(
        vocabulary=unique_movie_titles, mask_token=None
    ),
    tf.keras.layers.Embedding(len(unique_movie_titles) + 1, EMBEDDING_DIM)
], name="movie_tower")

#  Normaliser les embeddings des candidats
candidate_embeddings = movies.batch(128).map(
    lambda x: movie_model(x["movie_title"])
).cache() 

# Métrique
metrics = tfrs.metrics.FactorizedTopK(
    candidates=candidate_embeddings
)

# Tâche
task = tfrs.tasks.Retrieval(metrics=metrics)

# Modèle complet
class MovieRecommender(tfrs.Model):
    def __init__(self, user_model, movie_model, task):
        super().__init__()
        self.user_model = user_model
        self.movie_model = movie_model
        self.task = task
    
    def compute_loss(self, features, training=False):
        #  Extraire les features 
        user_embeddings = self.user_model(features["user_id"])
        movie_embeddings = self.movie_model(features["movie_title"])
        return self.task(user_embeddings, movie_embeddings)

model = MovieRecommender(user_model, movie_model, task)

In [32]:
# Hyperparamètres
LEARNING_RATE = 0.1
EPOCHS = 3
BATCH_SIZE = 8192

# run MLFlow
with mlflow.start_run(run_name=f"two-tower-v1-{datetime.now().strftime('%H%M%S')}"):
    
    # Logger les hyperparamètres
    mlflow.log_param("embedding_dim", EMBEDDING_DIM)
    mlflow.log_param("learning_rate", LEARNING_RATE)
    mlflow.log_param("epochs", EPOCHS)
    mlflow.log_param("batch_size", BATCH_SIZE)
    mlflow.log_param("optimizer", "Adagrad")
    mlflow.log_param("dataset", "MovieLens-100K")
    
    # Compiler
    model.compile(optimizer=tf.keras.optimizers.Adagrad(LEARNING_RATE))
    
    #  Pipeline de données
    cached_train_data = (
        train_data
        .shuffle(buffer_size=10_000, seed=42)
        .batch(BATCH_SIZE)
        .cache()
        .prefetch(tf.data.AUTOTUNE)
    )
    
    cached_test_data = (
        test_data
        .batch(BATCH_SIZE)
        .cache()
        .prefetch(tf.data.AUTOTUNE)
    )

    # Entraîner
    history = model.fit(
        cached_train_data,
        epochs=EPOCHS,
        verbose=1
    )
    
    # Logger les métriques
    for epoch, loss in enumerate(history.history['loss']):
        mlflow.log_metric("train_loss", loss, step=epoch)
        print(f"  Epoch {epoch+1}: Loss = {loss:.4f}")
    
    for epoch, acc in enumerate(history.history['factorized_top_k/top_100_categorical_accuracy']):
        mlflow.log_metric("top_100_accuracy", acc, step=epoch)
    
    # Évaluation
    test_metrics = model.evaluate(
        cached_test_data,
        return_dict=True
    )
    
    mlflow.log_metric("test_loss", test_metrics['loss'])
    mlflow.log_metric("test_top_100_accuracy", 
                      test_metrics['factorized_top_k/top_100_categorical_accuracy'])
    
    print(f"\nTest Loss: {test_metrics['loss']:.4f}")
    print(f"Test Accuracy: {test_metrics['factorized_top_k/top_100_categorical_accuracy']:.4f}")
    


Exception: Run with UUID 274d0f996a5140feb35faf68aa51d1ea is already active. To start a new run, first end the current run with mlflow.end_run(). To start a nested run, call start_run with nested=True

In [33]:
import pickle
import tempfile

# Sauvegarder et enregistrer la tour utilisateur
mlflow.tensorflow.log_model(
    model=model.user_model,
    artifact_path="user_tower",
    registered_model_name="MovieLens_UserTower"  # ← Cette ligne enregistre dans le Registry !
)

# Sauvegarder et enregistrer la tour film
mlflow.tensorflow.log_model(
    model=model.movie_model,
    artifact_path="movie_tower",
    registered_model_name="MovieLens_MovieTower"  # ← Cette ligne enregistre dans le Registry !
)

# Sauvegarder les vocabulaires
vocab_data = {
    'unique_user_ids': unique_user_ids.tolist(),
    'unique_movie_titles': unique_movie_titles.tolist()
}

with tempfile.TemporaryDirectory() as tmpdir:
    vocab_path = f"{tmpdir}/vocabularies.pkl"
    with open(vocab_path, 'wb') as f:
        pickle.dump(vocab_data, f)
    mlflow.log_artifact(vocab_path, artifact_path="vocabularies")

# Tags
mlflow.set_tag("model_type", "two-tower")
mlflow.set_tag("framework", "tensorflow-recommenders")
mlflow.set_tag("engineer", "Ahmed")

2025/11/16 15:58:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/16 15:58:58 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


2025-11-16 15:58:58 WARNI [tensorflow] Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.


INFO:tensorflow:Assets written to: C:\Users\ahmed\AppData\Local\Temp\tmp57w8ogos\model\data\model\assets


2025-11-16 15:58:58 INFO  [tensorflow] Assets written to: C:\Users\ahmed\AppData\Local\Temp\tmp57w8ogos\model\data\model\assets
2025/11/16 15:59:05 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/11/16 15:59:05 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2025/11/16 15:59:05 INFO mlflow.store.db.utils: Updating database tables
2025-11-16 15:59:05 INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
2025-11-16 15:59:05 INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
Successfully registered model 'MovieLens_UserTower'.
Created version '1' of model 'MovieLens_UserTower'.
2025/11/16 15:59:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/16 15:59:05 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference 

2025-11-16 15:59:05 WARNI [tensorflow] Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.


INFO:tensorflow:Assets written to: C:\Users\ahmed\AppData\Local\Temp\tmp3rh81onm\model\data\model\assets


2025-11-16 15:59:05 INFO  [tensorflow] Assets written to: C:\Users\ahmed\AppData\Local\Temp\tmp3rh81onm\model\data\model\assets
2025/11/16 15:59:11 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Successfully registered model 'MovieLens_MovieTower'.
Created version '1' of model 'MovieLens_MovieTower'.


pour l'interface mlflow, tu tapes mlflow ui sur le terminal puis tu peux checker les logs, les tours sont égalements sauvegardées ainsi que les données d'entrainement


In [34]:
mlflow.set_tracking_uri("sqlite:///../mlflow.db")

# Récupérer le dernier run
experiment = mlflow.get_experiment_by_name("news-recommender-two-tower")
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["start_time DESC"],
    max_results=1
)

if not runs.empty:
    run_id = runs.iloc[0]['run_id']
    print(f"Enregistrement depuis le run {run_id}...\n")
    
    # Enregistrer la tour utilisateur
    mlflow.register_model(
        model_uri=f"runs:/{run_id}/user_tower",
        name="MovieLens_UserTower"
    )
    
    # Enregistrer la tour film
    mlflow.register_model(
        model_uri=f"runs:/{run_id}/movie_tower",
        name="MovieLens_MovieTower"
    )
    print("✅ Tour film enregistrée")
    
    print("\n🎉 Rafraîchissez MLflow et allez dans l'onglet 'Models' !")

Registered model 'MovieLens_UserTower' already exists. Creating a new version of this model...
2025/11/16 15:59:20 WARNING mlflow.tracking._model_registry.fluent: Run with id 274d0f996a5140feb35faf68aa51d1ea has no artifacts at artifact path 'user_tower', registering model based on models:/m-e4fa015c03f741719db73d25cf9245e2 instead
Created version '2' of model 'MovieLens_UserTower'.
Registered model 'MovieLens_MovieTower' already exists. Creating a new version of this model...
2025/11/16 15:59:20 WARNING mlflow.tracking._model_registry.fluent: Run with id 274d0f996a5140feb35faf68aa51d1ea has no artifacts at artifact path 'movie_tower', registering model based on models:/m-53f7d8a0a84d48c2bb729b161baaecfd instead


Enregistrement depuis le run 274d0f996a5140feb35faf68aa51d1ea...

✅ Tour film enregistrée

🎉 Rafraîchissez MLflow et allez dans l'onglet 'Models' !


Created version '2' of model 'MovieLens_MovieTower'.
